# Silver Layer — Data Cleaning & Transformation
### Notebook: 02_silver_transformation
**Pipeline:** Bank Customer Churn Analytics
**Author:** Harshkumar Patel
**Date:** 29/06/2026

---

## What this notebook does
Silver layer makes the raw bronze data usable by cleaning it,
renaming columns and splitting it into a star schema structure

## Silver Layer Rule
> Clean it. Structure it. Enrich it. But do not aggregate it.

## Source
- **Database:** churn_bronze
- **Table:** churn_bronze.raw_bank_customers
- **Rows:** 80,000

## Target Tables
- **fact_churn_events** — 1 row per customer, central churn data
- **dim_customer** — customer demographics
- **dim_geography** — province and location info
- **dim_financial** — balance, credit score, cards
- **dim_engagement** — activity, engagement score, loyalty

## What we do in this notebook
1. Read from Bronze table
2. Rename truncated column names
3. Drop PII columns (full_name, address)
4. Fix data types
5. Remove duplicates
6. Feature engineering — create new columns
7. Split into star schema tables

## 1. Load Bronze Data

Read the raw customer data from the Bronze Delta table into a Spark DataFrame.

In [0]:
# loading data from the bronze delta table into silver for cleaning
df_bronze = spark.table("churn_bronze.raw_bank_customers")
print(f"Rows loaded from bronze table: {df_bronze.count():,}")

## 2. Remove PII & Unused Columns

Personal information is removed to protect customer privacy.
Pipeline metadata columns are also dropped since they are not needed for analysis.

In [0]:
# Check the number of columns before cleaning
print(f"Number of Columns: {len(df_bronze.columns)}")
# drop columns that are not needed for the analysis
df_clean = df_bronze.drop('full_name', 'address','_ingestion_timestamp', '_source_file', '_pipeline_layer')
print(f"Number of Columns after dropping: {len(df_clean.columns)}")

## 3. Rename Columns

Rename shortened column names to make the dataset easier to read and understand.

In [0]:
df_clean = df_clean.withColumnRenamed('tenure_ye', 'tenure_year')
df_clean = df_clean.withColumnRenamed('credit_sco', 'credit_score')
df_clean = df_clean.withColumnRenamed('monthly_ir', 'monthly_income')
df_clean = df_clean.withColumnRenamed('nums_card', 'num_of_cards')
df_clean = df_clean.withColumnRenamed('nums_service', 'num_of_services')

print(df_clean.columns)

## 4. Remove Duplicate Records

Ensure each customer appears only once using the customer ID.

In [0]:
# Count records before removing duplicates
print(df_clean.count())
# Remove duplicate customer IDs
df_clean = df_clean.dropDuplicates(['id'])
# Verify record count after cleaning
print(df_clean.count())

## 5. Fix Data Types

Convert columns to appropriate data types for accurate analysis.

In [0]:
from pyspark.sql.types import DoubleType
# Converting it to double because the column contains decimal values ot the balance can be in decimal values.
df_clean = df_clean.withColumn('balance', df_clean['balance'].cast(DoubleType()))

print(df_clean.printSchema())

# 6. Feature Engineering

Create additional business-friendly features that will help with reporting and dashboard analysis.

### Age Group & Balance Tier

Group customers into age categories for easier analysis.

Categorize customers based on account balance.

In [0]:
from pyspark.sql.functions import when

#Analyising the age column
print(df_clean.select('age').summary().display())
#Dividing age into 3 catogories
df_clean = df_clean.withColumn('age_group',
    when(df_clean['age'] <= 30, 'Young')
    .when(df_clean['age'] <= 50, 'Middle Aged')
    .otherwise('Senior')
)

df_clean.select('age', 'age_group').display()

#Analyising the balance column
print(df_clean.select('balance').summary().display())

#Dividing balance into 4 catogories

#important(
# dividing balance into tiers based on quartiles (25th, 50th, 75th percentile)
# this ensures customers are evenly distributed across tiers making analysis more meaningful)


df_clean = df_clean.withColumn('balance_tier',
    when(df_clean['balance'] <= 14622087, 'Low')
    .when(df_clean['balance'] <= 30896619, 'Medium')
    .when(df_clean['balance'] <= 69591695, 'High')
    .otherwise('Premium')
)


       
df_clean.select('balance', 'balance_tier').display()



### Tenure Group

Group customers based on how long they have been with the bank.

In [0]:
#Analyising the tenure column
print(df_clean.select('tenure_year').summary().display())
#Dividing tenure into 3 catogories
df_clean = df_clean.withColumn('tenure_group',
            when(df_clean['tenure_year'] <= 1, 'New')
            .when(df_clean['tenure_year'] <= 3, 'Regular')
            .otherwise('Loyal'))

df_clean.select('tenure_year', 'tenure_group').display()

df_clean.select('age', 'age_group').display()
df_clean.select('balance', 'balance_tier').display()
df_clean.select('tenure_year', 'tenure_group').display()

### Engagement & Churn Risk

Create engagement levels and a simple churn risk label using business rules.

In [0]:
#Analyising the Engagement column
df_clean.select('engagement_score').summary().display()

#Dividing engagement into 3 catogories
df_clean = df_clean.withColumn('engagement_tier',
            when(df_clean['engagement_score'] <= 25, 'Low')
            .when(df_clean['engagement_score'] <= 50, 'Medium')
            .otherwise('High'))

df_clean.select('engagement_score', 'engagement_tier').display()

In [0]:
#Analyising the risk_segment column
df_clean.select('risk_segment').distinct().display()
#not using this a sit oly has value low and medium

df_clean = df_clean.withColumn('churn_risk_label',
    when((df_clean['active_member'] == False) & (df_clean['engagement_tier'] == 'Low'), 'High Risk')
    .when((df_clean['active_member'] == True) & (df_clean['engagement_tier'] == 'High'), 'Low Risk')
    .otherwise('Medium Risk')
)

df_clean.select('churn_risk_label').display()


%md
## Why create a custom `churn_risk_label`?

According to the dataset documentation:

- `risk_score` estimates a combined financial or churn-related risk
- `risk_segment` is derived directly from that score

This means the existing risk features are **general-purpose risk indicators**, not specifically designed for customer churn behaviour.

However the existing risk_segment only contains Low and Medium values — there is no High Risk category, meaning highly disengaged customers are not being flagged.

For this project, the goal is to understand **churn driven by engagement and activity patterns**, so I created a separate behavioural risk label.

The `churn_risk_label` is built using:

- `active_member` (customer activity)
- `engagement_tier` (usage intensity)

This allows us to isolate **behavioral churn signals**, independent of financial risk.

For example:
- A customer may have low financial risk but still be at high churn risk due to inactivity
- Conversely, an active and engaged customer is less likely to churn regardless of financial profile

This feature helps complement the existing `risk_score` by adding a **behavior-first view of churn risk**, which is more relevant for retention analysis.

In the dashboard we can validate this label by checking whether customers labelled High Risk by our model actually have higher churn rates than those labelled Low Risk.

In [0]:
print(df_clean.columns)
print(f"total columns: {len(df_clean.columns)}")

In [0]:
df_clean.select("risk_score","risk_segment","churn_risk_label").display()


%md
## Dropping Irrelevant Columns

dropping columns that are not needed for churn analysis to reduce complexity
and keep the star schema clean

columns dropped and why:
- married — not relevant to churn behaviour
- last_active_date, last_transaction_month, created_date — date columns not needed for this analysis
- risk_segment — replaced by our own churn_risk_label
- digital_behavior, cluster_group — not useful for our KPIs
- loyalty_level — replaced by our own engagement_tier

In [0]:
df_clean = df_clean.drop(
    'married',
    'last_active_date',
    'last_transaction_month',
    'created_date',
    'risk_segment',
    'digital_behavior',
    'cluster_group',
    'loyalty_level'
)

print(f"columns remaining: {len(df_clean.columns)}")
print(df_clean.columns)

%md
## 7.Building the Star Schema

Now that the data is cleaned and features are engineered, we split df_clean 
into 5 separate tables following a Star Schema design.

Instead of keeping everything in one flat table with 21 columns, we separate 
it into 1 fact table and 4 dimension tables. This makes queries faster, 
the Power BI data model cleaner, and the analysis easier to maintain.

**Tables we are creating:**
- fact_churn_events — the core churn event, 1 row per customer
- dim_customer — who the customer is
- dim_scores — engagement and risk scores
- dim_customer_features — all engineered categorical features

In [0]:
# selecting columns for fact table - the core churn event
df_fact = df_clean.select(
    'id',
    'balance',
    'monthly_income',
    'tenure_year',
    'num_of_cards',
    'num_of_services',
    'active_member',
    'exit'
)

df_fact.write.format("delta").mode("overwrite").saveAsTable("churn_silver.fact_churn_events")

print(f"fact_churn_events rows: {df_fact.count():,}")

In [0]:
df_dim_customer = df_clean.select(
    "id",
    "credit_score",
    "gender",
    "age",
    "occupation",
    "origin_province",
    "customer_segment"
)

df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("churn_silver.dim_customer")

print(f"dim_customer rows: {df_dim_customer.count():,}")

In [0]:
df_dim_scores = df_clean.select(
    "id",
    "engagement_score",
    "risk_score"
)

df_dim_scores.write.format("delta").mode("overwrite").saveAsTable("churn_silver.dim_scores")

print(f"dim_scores rows: {df_dim_scores.count():,}")

In [0]:
df_dim_customer_features = df_clean.select(
    "id",
    "age_group",
    "balance_tier",
    "tenure_group",
    "engagement_tier",
    "churn_risk_label"
)

df_dim_customer_features.write.format("delta").mode("overwrite").saveAsTable("churn_silver.dim_customer_features")

print(f"dim_customer_features rows: {df_dim_customer_features.count():,}")

%md
## Silver Layer Complete

In this notebook we cleaned the raw bronze data, engineered 5 new behavioural 
features and split the data into a Star Schema of 4 tables:

| Table | Purpose |
|-------|---------|
| fact_churn_events | Core churn metrics — 1 row per customer |
| dim_customer | Customer demographics and location |
| dim_scores | Engagement and risk scores |
| dim_customer_features | Engineered features — age group, balance tier, risk label |

All 4 tables written to churn_silver database with 80,000 rows each.

**Next step:** Gold Layer — build KPI tables and churn aggregations using Spark SQL